# Inspect latest TripUpdates parquet

Use this notebook to quickly validate the latest TripUpdates parquet output written by the parse job.

This notebook:
- Lists parquet objects under `latest/TripUpdates/<agency>/`
- Loads either the newest file or a specific object path
- Shows schema and sample rows
- Runs lightweight quality checks

In [ ]:
from pathlib import Path
import tempfile

import pandas as pd
from google.cloud import storage

In [ ]:
# ----- Config -----
BUCKET = "511_transit_data"
FEED = "TripUpdates"
AGENCY = "muni"  # change to "bart" as needed
SERVICE_DATE = "2026-06-22"  # folder partition to inspect

# Optional: override full target prefix directly
# TARGET_PREFIX_OVERRIDE = "latest/TripUpdates/muni/2026-06-22/"
TARGET_PREFIX_OVERRIDE = None

In [ ]:
client = storage.Client()
if TARGET_PREFIX_OVERRIDE:
    target_prefix = TARGET_PREFIX_OVERRIDE
else:
    target_prefix = f"latest/{FEED}/{AGENCY}/{SERVICE_DATE}/"

blobs = [b for b in client.list_blobs(BUCKET, prefix=target_prefix) if b.name.endswith(".parquet")]
if not blobs:
    raise ValueError(f"No parquet files found in gs://{BUCKET}/{target_prefix}")

manifest = pd.DataFrame(
    {
        "blob_name": [b.name for b in blobs],
        "updated_utc": [b.updated for b in blobs],
        "size_bytes": [b.size for b in blobs],
    }
).sort_values("blob_name").reset_index(drop=True)

display(manifest)
print(f"Found {len(manifest):,} parquet shard files under gs://{BUCKET}/{target_prefix}")

In [ ]:
tmp_dir = Path(tempfile.mkdtemp(prefix="tripupdates_inspect_"))
local_paths = []

for object_path in manifest["blob_name"].tolist():
    local_path = tmp_dir / Path(object_path).name
    blob = client.bucket(BUCKET).blob(object_path)
    blob.download_to_filename(str(local_path))
    local_paths.append(local_path)

print(f"Downloaded {len(local_paths):,} shard files to: {tmp_dir}")

In [ ]:
df = pd.concat((pd.read_parquet(p) for p in local_paths), ignore_index=True)
print(f"Rows across all shards: {len(df):,}")
print("Columns:", list(df.columns))
display(df.head(20))

In [ ]:
print("Dtypes:")
display(df.dtypes)

print("\nNull counts:")
display(df.isna().sum().sort_values(ascending=False))

In [ ]:
# Expected key uniqueness check for latest-only output
expected_key_cols = [
    "agency_id",
    "trip_id",
    "trip_start_date",
    "trip_start_time",
    "direction_id",
    "stop_sequence",
]

missing_key_cols = [c for c in expected_key_cols if c not in df.columns]
if missing_key_cols:
    print("Missing expected key columns:", missing_key_cols)
else:
    dupes = df.duplicated(subset=expected_key_cols, keep=False).sum()
    print(f"Duplicated rows on expected latest key: {dupes:,}")
    print(f"Distinct key count: {df[expected_key_cols].drop_duplicates().shape[0]:,}")